# DiffInfinite - NIPA 병리 이미지 학습

NIPA 병리 패치 이미지(1024x1024, JPEG)를 이용한 DiffInfinite 모델 학습  
- 데이터: `../../data/NIPA/origin/` (클래스별 폴더)
- 모델 저장: `../../model/NIPA/diffinfinite/`

## 1. 환경 설정 및 Import

In [ ]:
import os
import sys
import math
import random
from pathlib import Path
from glob import glob

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, lr_scheduler
import torchvision.transforms as T
from torchvision import utils as vutils
from PIL import Image
from tqdm import tqdm
from ema_pytorch import EMA
from accelerate import Accelerator
from diffusers import AutoencoderKL

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from dm import Unet, GaussianDiffusion
from utils.helpers import exists, cycle, has_int_squareroot
from dataset import ComposeState, RandomRotate90

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "CPU only")

## 2. 설정값 정의

In [ ]:
# 경로
DATA_ROOT = "../../../data/NIPA/origin"
MODEL_SAVE_DIR = "../../../model/NIPA/diffinfinite"
RESULTS_DIR = os.path.join(MODEL_SAVE_DIR, "results")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# 모델 하이퍼파라미터
IMAGE_SIZE = 1024          # 1024x1024 원본 그대로 사용
Z_SIZE = IMAGE_SIZE // 8   # VAE latent size = 128
DIM = 256
DIM_MULTS = (1, 2, 4)
CHANNELS = 4               # VAE latent channels
RESNET_BLOCK_GROUPS = 2
BLOCK_PER_LAYER = 2
TIMESTEPS = 1000
SAMPLING_TIMESTEPS = 250

# 학습 하이퍼파라미터
BATCH_SIZE = 1             # 1024x1024 -> latent 128x128, VRAM 절약
LEARNING_RATE = 1e-4
TRAIN_NUM_STEPS = 25000
SAVE_SAMPLE_EVERY = 5000
SAVE_LOSS_EVERY = 100
GRADIENT_ACCUMULATE_EVERY = 4  # 실질 batch size = 4
NUM_SAMPLES = 4
NUM_WORKERS = 4
TRAIN_RATIO = 0.9

# 클래스 정보
CLASS_NAMES = sorted(os.listdir(DATA_ROOT))
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

print(f"클래스 수: {NUM_CLASSES} (unconditional 포함)")
print(f"클래스 목록: {CLASS_TO_IDX}")
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Latent size: {Z_SIZE}x{Z_SIZE}")
print(f"Effective batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATE_EVERY} = {BATCH_SIZE * GRADIENT_ACCUMULATE_EVERY}")

In [ ]:
CLASS_NAMES

## 3. 데이터셋 정의

클래스별 폴더 구조에서 이미지를 로드하고, DiffInfinite가 요구하는 형식(이미지 + 마스크)으로 변환합니다.  
- 세그멘테이션 마스크가 없으므로 클래스 레이블로 채운 uniform mask를 생성합니다.
- 1024x1024 이미지를 그대로 사용합니다.

In [ ]:
class NIPAPathologyDataset(Dataset):
    """NIPA 병리 패치 이미지 데이터셋 (클래스별 폴더 구조)"""
    
    def __init__(self, file_list, labels, image_size=1024, cond_drop_prob=0.5, is_train=True):
        self.file_list = file_list
        self.labels = labels
        self.image_size = image_size
        self.cond_drop_prob = cond_drop_prob
        self.is_train = is_train
        
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.RandomHorizontalFlip() if is_train else T.Lambda(lambda x: x),
            T.RandomVerticalFlip() if is_train else T.Lambda(lambda x: x),
        ])
    
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        label = self.labels[idx]
        
        # Classifier-free guidance: 일정 확률로 unconditional (label=0)
        if self.is_train and random.random() < self.cond_drop_prob:
            label = 0
        
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        
        # 클래스 레이블로 채운 마스크 생성 (1, H, W)
        mask = torch.full((1, self.image_size, self.image_size), label, dtype=torch.long)
        
        return img, mask


def build_datasets(data_root, class_to_idx, image_size=1024, train_ratio=0.9, cond_drop_prob=0.5):
    """클래스별 폴더에서 train/val 데이터셋 생성"""
    all_files = []
    all_labels = []
    
    for class_name, class_idx in class_to_idx.items():
        class_dir = os.path.join(data_root, class_name)
        if not os.path.isdir(class_dir):
            continue
        
        img_files = sorted(glob(os.path.join(class_dir, "*.jpeg")) + 
                          glob(os.path.join(class_dir, "*.jpg")) +
                          glob(os.path.join(class_dir, "*.JPEG")))
        
        all_files.extend(img_files)
        all_labels.extend([class_idx] * len(img_files))
        print(f"  {class_name} (idx={class_idx}): {len(img_files)} images")
    
    # 셔플 및 분할
    indices = list(range(len(all_files)))
    random.seed(42)
    random.shuffle(indices)
    
    split_idx = int(len(indices) * train_ratio)
    train_indices = indices[:split_idx]
    val_indices = indices[split_idx:]
    
    train_files = [all_files[i] for i in train_indices]
    train_labels = [all_labels[i] for i in train_indices]
    val_files = [all_files[i] for i in val_indices]
    val_labels = [all_labels[i] for i in val_indices]
    
    train_dataset = NIPAPathologyDataset(
        train_files, train_labels, image_size=image_size, 
        cond_drop_prob=cond_drop_prob, is_train=True
    )
    val_dataset = NIPAPathologyDataset(
        val_files, val_labels, image_size=image_size, 
        cond_drop_prob=1.0, is_train=False
    )
    
    print(f"\nTotal: {len(all_files)} images")
    print(f"Train: {len(train_files)}, Val: {len(val_files)}")
    
    return train_dataset, val_dataset

## 4. 데이터셋 생성 및 확인

In [ ]:
train_dataset, val_dataset = build_datasets(
    DATA_ROOT, CLASS_TO_IDX, 
    image_size=IMAGE_SIZE, 
    train_ratio=TRAIN_RATIO,
    cond_drop_prob=0.5
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)

# 샘플 확인
sample_img, sample_mask = train_dataset[0]
print(f"Image shape: {sample_img.shape}, dtype: {sample_img.dtype}")
print(f"Mask shape: {sample_mask.shape}, dtype: {sample_mask.dtype}")
print(f"Mask unique values: {torch.unique(sample_mask)}")
print(f"Image value range: [{sample_img.min():.3f}, {sample_img.max():.3f}]")

In [ ]:
# 샘플 이미지 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(8):
    img, mask = train_dataset[i * 100]
    ax = axes[i // 4][i % 4]
    ax.imshow(img.permute(1, 2, 0).numpy())
    label_val = mask[0, 0, 0].item()
    class_name = [k for k, v in CLASS_TO_IDX.items() if v == label_val]
    class_name = class_name[0] if class_name else "uncond"
    ax.set_title(f"{class_name} (label={label_val})")
    ax.axis("off")
plt.suptitle("Train Dataset Samples", fontsize=14)
plt.tight_layout()
plt.show()

## 5. 모델 선언

- **VAE**: Stable Diffusion 2 VAE (이미지를 latent space로 인코딩)
- **UNet**: DiffInfinite UNet (latent space에서 diffusion 수행)
- **GaussianDiffusion**: diffusion process 관리

In [ ]:
# UNet 생성
unet = Unet(
    dim=DIM,
    num_classes=NUM_CLASSES,
    dim_mults=DIM_MULTS,
    channels=CHANNELS,
    resnet_block_groups=RESNET_BLOCK_GROUPS,
    block_per_layer=BLOCK_PER_LAYER,
)

# GaussianDiffusion 모델
diffusion_model = GaussianDiffusion(
    unet,
    image_size=Z_SIZE,
    timesteps=TIMESTEPS,
    sampling_timesteps=SAMPLING_TIMESTEPS,
    loss_type='l2',
)

# VAE (Stable Diffusion VAE - 인증 불필요)
print("Loading VAE from stabilityai/sd-vae-ft-mse...")
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
vae.eval()
for param in vae.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in diffusion_model.parameters() if p.requires_grad)
print(f"\nUNet trainable parameters: {total_params:,}")
print(f"Model image_size (latent): {Z_SIZE}")
print(f"Num classes: {NUM_CLASSES}")

## 6. 학습 설정 (Accelerator, Optimizer, Scheduler, EMA)

In [ ]:
# Accelerator 설정
accelerator = Accelerator(
    split_batches=True,
    mixed_precision='fp16',
    gradient_accumulation_steps=GRADIENT_ACCUMULATE_EVERY,
)

# Optimizer
optimizer = Adam(diffusion_model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.99))

# LR Scheduler
scheduler = lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE, total_steps=TRAIN_NUM_STEPS
)

# EMA
ema = EMA(diffusion_model, beta=0.995, update_every=10)

# Accelerator에 모델, 옵티마이저, 데이터로더 등록
diffusion_model, optimizer, ema, scheduler, train_loader, val_loader = accelerator.prepare(
    diffusion_model, optimizer, ema, scheduler, train_loader, val_loader
)

vae = accelerator.prepare(vae)

# 무한 반복 데이터로더
train_dl = cycle(train_loader)
val_dl = cycle(val_loader)

print("Training setup complete!")

## 7. 학습 함수 및 저장 함수 정의

In [ ]:
def get_vae_module(vae_model):
    """accelerator wrapping에 따라 실제 vae 모듈 반환"""
    if hasattr(vae_model, 'module'):
        return vae_model.module
    return vae_model


def save_checkpoint(step, milestone, diffusion_model, optimizer, scheduler, ema, 
                    running_loss, running_lr, save_dir):
    """모델 체크포인트 저장"""
    data = {
        'step': step,
        'loss': running_loss,
        'lr': running_lr,
        'model': accelerator.get_state_dict(diffusion_model),
        'opt': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'ema': accelerator.get_state_dict(ema),
        'scaler': accelerator.scaler.state_dict() if exists(accelerator.scaler) else None,
        'config': {
            'dim': DIM,
            'num_classes': NUM_CLASSES,
            'dim_mults': DIM_MULTS,
            'channels': CHANNELS,
            'image_size': IMAGE_SIZE,
            'z_size': Z_SIZE,
            'class_to_idx': CLASS_TO_IDX,
        }
    }
    save_path = os.path.join(save_dir, f'model-{milestone}.pt')
    torch.save(data, save_path)
    print(f"  Checkpoint saved: {save_path}")


def load_checkpoint(milestone, diffusion_model, optimizer, scheduler, ema, save_dir):
    """체크포인트 로드"""
    load_path = os.path.join(save_dir, f'model-{milestone}.pt')
    data = torch.load(load_path, map_location=accelerator.device)
    
    model = accelerator.unwrap_model(diffusion_model)
    model.load_state_dict(data['model'])
    
    optimizer.load_state_dict(data['opt'])
    scheduler.load_state_dict(data['scheduler'])
    
    ema_unwrapped = accelerator.unwrap_model(ema)
    ema_unwrapped.load_state_dict(data['ema'])
    
    if exists(data.get('scaler')) and exists(accelerator.scaler):
        accelerator.scaler.load_state_dict(data['scaler'])
    
    print(f"  Checkpoint loaded from step {data['step']}")
    return data['step'], data.get('loss', []), data.get('lr', [])


def generate_samples(diffusion_model, ema, vae_model, val_dl, num_samples, step, save_dir):
    """검증 데이터로 샘플 생성 및 저장"""
    ema_unwrapped = accelerator.unwrap_model(ema) if hasattr(ema, 'module') else ema
    
    if hasattr(ema_unwrapped, 'module'):
        ema_model = ema_unwrapped.module.ema_model
    elif hasattr(ema_unwrapped, 'ema_model'):
        ema_model = ema_unwrapped.ema_model
    else:
        ema_model = ema_unwrapped
    
    ema_model.eval()
    vae_mod = get_vae_module(vae_model)
    
    with torch.no_grad():
        test_images, test_masks = next(val_dl)
        test_images = test_images[:num_samples]
        test_masks = test_masks[:num_samples]
        
        # 원본 이미지를 latent로 인코딩
        z = vae_mod.encode(test_images).latent_dist.sample() / 50
        
        # EMA 모델로 샘플 생성
        z_sampled = ema_model.sample(z, test_masks) * 50
        
        # latent에서 이미지로 디코딩
        samples = torch.clip(vae_mod.decode(z_sampled).sample, 0, 1)
    
    milestone = step // SAVE_SAMPLE_EVERY
    nrow = int(math.sqrt(num_samples))
    
    vutils.save_image(
        test_images, os.path.join(save_dir, f'original-{milestone}.png'), nrow=nrow
    )
    vutils.save_image(
        samples, os.path.join(save_dir, f'sample-{milestone}.png'), nrow=nrow
    )
    
    print(f"  Samples saved at step {step}")

print("Functions defined.")

## 8. 학습 실행

체크포인트에서 이어서 학습하려면 아래 `RESUME_MILESTONE`을 설정하세요.

In [ ]:
# 이어서 학습할 경우 milestone 번호 설정 (None이면 처음부터)
RESUME_MILESTONE = None

step = 0
running_loss = []
running_lr = []

if RESUME_MILESTONE is not None:
    step, running_loss, running_lr = load_checkpoint(
        RESUME_MILESTONE, diffusion_model, optimizer, scheduler, ema, MODEL_SAVE_DIR
    )

vae_mod = get_vae_module(vae)

with tqdm(initial=step, total=TRAIN_NUM_STEPS, disable=not accelerator.is_main_process) as pbar:
    while step < TRAIN_NUM_STEPS:
        total_loss = 0.0
        
        for _ in range(GRADIENT_ACCUMULATE_EVERY):
            imgs, masks = next(train_dl)
            
            with accelerator.accumulate(diffusion_model):
                # VAE로 latent encoding
                with torch.no_grad():
                    latents = vae_mod.encode(imgs).latent_dist.sample() / 50
                
                # Diffusion loss 계산
                with accelerator.autocast():
                    loss = diffusion_model(img=latents, classes=masks)
                
                accelerator.backward(loss)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                
                total_loss += loss.item()
        
        total_loss /= GRADIENT_ACCUMULATE_EVERY
        
        # Loss/LR 기록
        if step % SAVE_LOSS_EVERY == 0:
            running_loss.append(total_loss)
            running_lr.append(scheduler.get_lr()[0])
        
        pbar.set_description(f"loss: {total_loss:.4f}")
        
        # EMA 업데이트
        if accelerator.is_main_process:
            ema_unwrapped = accelerator.unwrap_model(ema) if hasattr(ema, 'module') else ema
            if hasattr(ema_unwrapped, 'module'):
                ema_unwrapped.module.update()
            elif hasattr(ema_unwrapped, 'update'):
                ema_unwrapped.update()
        
        step += 1
        
        # 샘플 생성 및 체크포인트 저장
        if step % SAVE_SAMPLE_EVERY == 0 and accelerator.is_main_process:
            milestone = step // SAVE_SAMPLE_EVERY
            print(f"\n--- Step {step} (Milestone {milestone}) ---")
            
            generate_samples(
                diffusion_model, ema, vae, val_dl, NUM_SAMPLES, step, RESULTS_DIR
            )
            save_checkpoint(
                step, milestone, diffusion_model, optimizer, scheduler, ema,
                running_loss, running_lr, MODEL_SAVE_DIR
            )
        
        pbar.update(1)

print("Training complete!")

## 9. 학습 결과 시각화

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss 그래프
ax1.plot(running_loss)
ax1.set_title("Training Loss")
ax1.set_xlabel(f"Step (x{SAVE_LOSS_EVERY})")
ax1.set_ylabel("Loss")
ax1.set_yscale("log")
ax1.grid(True, alpha=0.3)

# Learning Rate 그래프
ax2.plot(running_lr)
ax2.set_title("Learning Rate")
ax2.set_xlabel(f"Step (x{SAVE_LOSS_EVERY})")
ax2.set_ylabel("LR")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Final loss: {running_loss[-1]:.6f}" if running_loss else "No loss recorded yet")

## 10. 생성된 샘플 이미지 확인

In [ ]:
# 가장 최근 생성된 샘플 이미지 표시
sample_files = sorted(glob(os.path.join(RESULTS_DIR, "sample-*.png")))
original_files = sorted(glob(os.path.join(RESULTS_DIR, "original-*.png")))

if sample_files and original_files:
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    orig_img = Image.open(original_files[-1])
    axes[0].imshow(orig_img)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    sample_img = Image.open(sample_files[-1])
    axes[1].imshow(sample_img)
    axes[1].set_title("Generated")
    axes[1].axis("off")
    
    plt.suptitle(f"Latest Sample (Milestone {len(sample_files)})", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No sample images found yet. Run training first.")

## 11. 최종 모델 저장

In [ ]:
# 최종 모델 저장
final_data = {
    'step': step,
    'loss': running_loss,
    'lr': running_lr,
    'model': accelerator.get_state_dict(diffusion_model),
    'ema': accelerator.get_state_dict(ema),
    'config': {
        'dim': DIM,
        'num_classes': NUM_CLASSES,
        'dim_mults': DIM_MULTS,
        'channels': CHANNELS,
        'resnet_block_groups': RESNET_BLOCK_GROUPS,
        'block_per_layer': BLOCK_PER_LAYER,
        'image_size': IMAGE_SIZE,
        'z_size': Z_SIZE,
        'timesteps': TIMESTEPS,
        'sampling_timesteps': SAMPLING_TIMESTEPS,
        'class_names': CLASS_NAMES,
        'class_to_idx': CLASS_TO_IDX,
    }
}

final_path = os.path.join(MODEL_SAVE_DIR, "model-final.pt")
torch.save(final_data, final_path)
print(f"Final model saved: {final_path}")
print(f"Total training steps: {step}")
print(f"Model save directory: {os.path.abspath(MODEL_SAVE_DIR)}")